# Growing Sensor Analysis

This notebook isolates the growing sensor analysis from the compost analysis and adds:

- sensor-level uptime and zero-rate profiling,
- rolling active-sensor tracking,
- sensor timelines,
- correlation analysis using active readings only, and
- a weekly moisture heatmap to spot persistent gaps.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)


In [ ]:
file_path = "../data/combined_compost_measurements.csv"

df = pd.read_csv(file_path, parse_dates=["Day"])

sensor_df = (
    df.rename(
        columns={
            "Day": "day",
            "Growing-Sensor 01-Moisture": "Sensor 01",
            "Growing-Sensor 02-Moisture": "Sensor 02",
            "Growing-Sensor 03-Moisture": "Sensor 03",
            "Growing-Sensor 04-Moisture": "Sensor 04",
            "Growing-Sensor 05-Moisture": "Sensor 05",
            "Growing-Sensor 06-Moisture": "Sensor 06",
            "Growing-Sensor 07-Moisture": "Sensor 07",
            "Growing-Sensor 08-Moisture": "Sensor 08",
            "Growing-Sensor 09-Moisture": "Sensor 09",
            "Growing-Sensor 10-Moisture": "Sensor 10",
            "Growing-Sensor 11-Moisture": "Sensor 11",
            "Growing-Sensor 12-Moisture": "Sensor 12",
        }
    )[
        [
            "day",
            "Sensor 01",
            "Sensor 02",
            "Sensor 03",
            "Sensor 04",
            "Sensor 05",
            "Sensor 06",
            "Sensor 07",
            "Sensor 08",
            "Sensor 09",
            "Sensor 10",
            "Sensor 11",
            "Sensor 12",
        ]
    ]
    .sort_values("day")
    .reset_index(drop=True)
)

sensor_df


## Reliability Profile

A zero reading can mean either truly dry soil or a dead/idle sensor. The summary below treats zeros as a reliability signal first, then shows the average moisture only when the sensor is actively reporting above zero.


In [ ]:
sensor_pd = sensor_df.copy()
sensor_cols = [column for column in sensor_pd.columns if column != "day"]

def longest_zero_streak(series: pd.Series) -> int:
    longest = 0
    current = 0

    for is_zero in series.fillna(0).eq(0):
        if is_zero:
            current += 1
            longest = max(longest, current)
        else:
            current = 0

    return longest

sensor_summary = pd.DataFrame(
    {
        "zero_share": (sensor_pd[sensor_cols] == 0).mean().round(3),
        "active_share": (sensor_pd[sensor_cols] > 0).mean().round(3),
        "mean_when_active": sensor_pd[sensor_cols].replace(0, np.nan).mean().round(2),
        "longest_zero_streak_days": [longest_zero_streak(sensor_pd[column]) for column in sensor_cols],
    }
).sort_values(["zero_share", "longest_zero_streak_days"], ascending=False)

sensor_summary


In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(
    data=sensor_summary.reset_index(),
    x="index",
    y="zero_share",
    hue="index",
    palette="crest",
    legend=False,
)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Share of zero readings")
plt.xlabel("Sensor")
plt.title("Zero-rate ranking across growing sensors")
plt.tight_layout()
plt.show()


## How Many Sensors Are Active At Once?

This helps distinguish between individual failures and broader downtime events that hit many sensors at the same time.


In [ ]:
sensor_pd["active_sensor_count"] = (sensor_pd[sensor_cols] > 0).sum(axis=1)
sensor_pd["active_sensor_count_7d"] = sensor_pd["active_sensor_count"].rolling(7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(sensor_pd["day"], sensor_pd["active_sensor_count"], color="tab:gray", alpha=0.35, label="Daily active count")
ax.plot(sensor_pd["day"], sensor_pd["active_sensor_count_7d"], color="tab:green", linewidth=2.5, label="7d rolling active count")
ax.set_title("Number of growing sensors reporting above zero")
ax.set_xlabel("Day")
ax.set_ylabel("Active sensors")
ax.set_ylim(0, len(sensor_cols) + 0.5)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()


In [ ]:
long_sensor = sensor_pd.melt(
    id_vars="day",
    value_vars=sensor_cols,
    var_name="sensor",
    value_name="moisture",
)

sensor_grid = sns.relplot(
    data=long_sensor,
    x="day",
    y="moisture",
    col="sensor",
    col_wrap=4,
    kind="line",
    height=3,
    aspect=1.2,
    facet_kws={"sharey": False},
)
sensor_grid.figure.subplots_adjust(top=0.93)
sensor_grid.figure.suptitle("Per-sensor moisture timelines")
plt.show()


Looking at the individual timelines is useful for spotting partial recoveries. Some sensors can resume normal variation after a shared outage, while others remain flat and likely need a hardware check.


In [ ]:
active_only_corr = sensor_pd[sensor_cols].replace(0, np.nan).corr()

plt.figure(figsize=(12, 10))
sns.heatmap(active_only_corr, annot=True, cmap="magma", center=0, fmt=".2f")
plt.title("Sensor correlation using active readings only")
plt.tight_layout()
plt.show()


In [ ]:
weekly_sensor = long_sensor.copy()
weekly_sensor["moisture"] = weekly_sensor["moisture"].replace(0, np.nan)
weekly_sensor["week"] = weekly_sensor["day"].dt.to_period("W").apply(lambda value: value.start_time)

weekly_heatmap = weekly_sensor.pivot_table(
    index="sensor",
    columns="week",
    values="moisture",
    aggfunc="median",
)

plt.figure(figsize=(16, 6))
sns.heatmap(weekly_heatmap, cmap="YlGnBu", linewidths=0.2, linecolor="white")
plt.title("Weekly median moisture by sensor")
plt.xlabel("Week")
plt.ylabel("Sensor")
plt.tight_layout()
plt.show()


## Takeaways

- The sensor analysis now stands on its own and is easier to review separately from compost temperature behaviour.
- Zero-rate ranking and longest-streak summaries make hardware issues easier to prioritize.
- Active-sensor counts expose shared downtime windows that are hard to see from one sensor at a time.
- The weekly heatmap gives a compact view of where recovery happens and which sensors remain persistently silent.
